# ML3 P300 XGBoost — Colab GPU Runner & Publication Visualizations

**Cách chạy trên VS Code (Google Colab Extension):**
1. Bấm **Select Kernel** (góc trên bên phải) $\rightarrow$ **Colab** $\rightarrow$ chọn GPU (T4 / L4).
2. Nhấn `Ctrl+Shift+P` $\rightarrow$ chọn **`Colab: Mount Google Drive to Server...`** để gắn `/content/drive`.
3. Chạy lần lượt các ô bên dưới (dữ liệu `features/` được copy sang SSD `/content/features` để đọc siêu nhanh; checkpoint & toàn bộ biểu đồ chuẩn `setup_publication_style()` + `OKABE_ITO` lưu về Google Drive và hiển thị trực tiếp trong notebook).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
# 1. Kiểm tra GPU, đồng bộ code mới nhất vào /content/EEG-mini-project và cài xgboost
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_GIT_URL = "https://github.com/Longkyoladn/EEG-mini-project.git"
REPO_BRANCH = "main"
COLAB_REPO_DIR = Path("/content/EEG-mini-project")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/EEG-mini-project")
LOCAL_FEATURES_DIR = Path("/content/features")

subprocess.run(["nvidia-smi"], check=False)

if not COLAB_REPO_DIR.exists():
    if (DRIVE_PROJECT_DIR / "src").is_dir() and (DRIVE_PROJECT_DIR / "splits").is_dir():
        print(f"Copying repo code from Drive {DRIVE_PROJECT_DIR} -> {COLAB_REPO_DIR} ...")
        shutil.copytree(
            DRIVE_PROJECT_DIR,
            COLAB_REPO_DIR,
            ignore=shutil.ignore_patterns("features", "results", "data", "oddball_data", ".git"),
        )
    else:
        subprocess.run(
            ["git", "clone", "-b", REPO_BRANCH, REPO_GIT_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
elif (COLAB_REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(COLAB_REPO_DIR), "pull"], check=False)

os.chdir(COLAB_REPO_DIR)
if str(COLAB_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(COLAB_REPO_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0.0"], check=True)
print(f"Ready at: {Path.cwd()}")

Ready at: /content/EEG-mini-project


In [3]:
# 2. Kiểm tra Google Drive & copy features/temporal + features/wavelet sang /content/features
# (Các file đã copy sẵn trong /content/features sẽ được giữ nguyên, không copy lại)
from scripts.training.ML3.train_ml3 import main as train_ml3_main, setup_colab_drive_paths

paths = setup_colab_drive_paths(
    drive_project_dir=DRIVE_PROJECT_DIR,
    stage_to_content=True,
    local_features_dir=LOCAL_FEATURES_DIR,
)
for k, v in paths.items():
    print(f"{k}: {v}")

[Stage Features] /content/drive/MyDrive/EEG-mini-project/features -> /content/features | copied=78, reused=0, transferred=962.5 MB
project_dir: /content/drive/MyDrive/EEG-mini-project
drive_features_dir: /content/drive/MyDrive/EEG-mini-project/features
features_dir: /content/features
temporal_dir: /content/features/temporal
wavelet_dir: /content/features/wavelet
output_dir: /content/drive/MyDrive/EEG-mini-project/results/ml3


In [ ]:
# 3. Kiểm tra nhanh cấu hình & số lượng feature (--plan-only)
train_ml3_main(
    [
        "--features-dir", str(paths["features_dir"]),
        "--splits-dir", str(COLAB_REPO_DIR / "splits"),
        "--protocol", "cs5",
        "--output-dir", str(paths["output_dir"]),
        "--device", "cuda",
        "--plan-only",
    ]
)

In [ ]:
# 4. Chạy Cross-Subject 5-Fold (5 kịch bản chính)
train_ml3_main(
    [
        "--features-dir", str(paths["features_dir"]),
        "--splits-dir", str(COLAB_REPO_DIR / "splits"),
        "--protocol", "cs5",
        "--experiments", "temporal", "wavelet_w5", "wavelet_w8", "combined_w5", "combined_w8",
        "--output-dir", str(paths["output_dir"]),
        "--device", "cuda",
        "--resume",
    ]
)

In [ ]:
# 5. Chạy LOSO (3 kịch bản chính: temporal, combined_w5, combined_w8)
train_ml3_main(
    [
        "--features-dir", str(paths["features_dir"]),
        "--splits-dir", str(COLAB_REPO_DIR / "splits"),
        "--protocol", "loso",
        "--experiments", "temporal", "combined_w5", "combined_w8",
        "--output-dir", str(paths["output_dir"]),
        "--device", "cuda",
        "--resume",
    ]
)

In [ ]:
# 6. (Tùy chọn) Kiểm tra bổ sung Gamma trên CS5
# Đổi "combined_w5_gamma" thành "combined_w8_gamma" nếu W8 là cấu hình cần kiểm chứng
# train_ml3_main(
#     [
#         "--features-dir", str(paths["features_dir"]),
#         "--splits-dir", str(COLAB_REPO_DIR / "splits"),
#         "--protocol", "cs5",
#         "--experiments", "combined_w5_gamma",
#         "--output-dir", str(paths["output_dir"]),
#         "--device", "cuda",
#         "--resume",
#     ]
# )

---
## 7. Vẽ toàn bộ biểu đồ chuẩn bài báo (`setup_publication_style()` + `OKABE_ITO`, 300 DPI) & Chẩn đoán Overfitting
- Dùng bản tối ưu dựng `full_dm` 1 lần/kịch bản trên GPU (chỉ mất ~20–40 giây cho các fold cũ chưa có `learning_curve.json`, từ lần thứ 2 trở đi chỉ mất ~3 giây).
- Ảnh tự động lưu an toàn (atomic write) lên Google Drive tại `/content/drive/MyDrive/EEG-mini-project/results/ml3/{cs5|loso}/` và hiển thị trực tiếp ngay bên dưới.

In [ ]:
# 7. Tái tạo learning curve (bản tối ưu GPU full_dm) + Vẽ & hiển thị toàn bộ biểu đồ 300 DPI
import importlib
import pandas as pd
from IPython.display import Image, Markdown, display

import src.models.Ml3 as ml3_mod
import src.training.ml3_training as ml3_trn

importlib.reload(ml3_mod)
importlib.reload(ml3_trn)
ml3_trn.setup_publication_style()

# Chọn giao thức muốn vẽ: ("cs5",) hoặc ("cs5", "loso")
PROTOCOLS_TO_PLOT = ("cs5", "loso")

for proto in PROTOCOLS_TO_PLOT:
    proto_dir = Path(paths["output_dir"]) / proto
    if not proto_dir.is_dir():
        print(f"[Skip] Chưa có thư mục kết quả cho {proto.upper()}: {proto_dir}")
        continue

    n_backfilled = ml3_trn.backfill_learning_curves_from_checkpoints(
        protocol_dir=proto_dir,
        features_dir=paths["features_dir"],
        splits_dir=COLAB_REPO_DIR / "splits",
        protocol=proto,
        device="cuda",
        n_curve_points=20,
        verbose=True,
    )
    if n_backfilled:
        print(f"[{proto.upper()}] Đã bổ sung learning_curve.json cho {n_backfilled} fold.")

    comp = ml3_trn.write_protocol_comparison_and_plots(
        proto,
        proto_dir,
        features_dir=paths["features_dir"],
    )
    if not comp:
        continue

    display(Markdown(f"### Bảng tổng hợp {proto.upper()} (`comparison_table.csv`)"))
    display(pd.read_csv(proto_dir / "comparison_table.csv"))

    for plot_name in comp.get("plots", []):
        img_path = proto_dir / plot_name
        if img_path.is_file():
            display(Markdown(f"#### `{proto}/{plot_name}`"))
            display(Image(filename=str(img_path), width=980))